In [20]:
from pathlib import Path
import ims
import preprocess

# Step 0: Define all necessary directories.
input_dir = Path("./data/IMS_Olive_Oil_Geography/Greece/O-034")
plot_dir = Path("./output/spectra/")
output_dir = Path("./output")

# Check and create directories if necessary.
if not plot_dir.exists():
    plot_dir.mkdir(parents=True, exist_ok=True)
if not output_dir.exists():
    output_dir.mkdir(parents=True, exist_ok=True)

# Step 1: Preprocess the .mea files
# - RIP Interpolation (for each dataset)
# - Background noise reduction
# - Gaussian smoothing

# Load the test dataset and interpolate to a common RIP
dataset = ims.Dataset.read_mea(input_dir)
dataset.interp_riprel()

# Get the first test spectrum.
spectrum = dataset[0]

In [2]:
spectrum.values

array([[34.        , 32.998924  , 53.97130499, ..., 16.98673404,
        10.00256667, 15.99690467],
       [13.        , 15.99547755, 45.96953196, ..., 26.05384757,
        54.0048843 , 34.98247033],
       [30.        , 20.9988723 , 40.97563597, ..., 18.99773859,
        14.00237959, 25.01684307],
       ...,
       [49.        , 33.00336077, 48.96626517, ..., 19.9595465 ,
        22.04478441, 48.99431901],
       [48.        , 15.99426207, 63.95694002, ..., 17.01316638,
        27.01638684, 36.01116142],
       [22.        , 32.9928232 , 64.97442862, ..., 28.00717244,
        25.99498732, 24.99983241]], shape=(6939, 3147))

In [21]:
# Preprocess the spectrum...
drift_cut= (1.02, 2.5)
retention_cut = (80, 900)

# Remove background noise and cut the spectrum
spectrum.sub_first_rows(50)
spectrum.cut_dt(start=drift_cut[0], stop=drift_cut[1])
spectrum.cut_rt(start=retention_cut[0], stop=retention_cut[1])

GC-IMS Spectrum: 151207_144657

In [22]:
# Smooth the spectrum and set negative intensities to 0.0.
#spectrum.savgol(window_length=20, polyorder=2)
spectrum.values[spectrum.values < 0] = 0.0
spectrum.savgol(window_length=20, polyorder=2)
spectrum.values[spectrum.values < 0] = 0.0
spectrum.values

array([[0.        , 0.        , 0.18108707, ..., 2.9978823 , 4.24465867,
        5.69626867],
       [0.        , 0.        , 0.42165576, ..., 3.52533756, 4.58075576,
        5.80522689],
       [0.        , 0.        , 0.67522459, ..., 3.99031813, 4.86750678,
        5.88071864],
       ...,
       [3.39528483, 4.00092947, 4.53910882, ..., 5.3263674 , 5.24736084,
        5.1549786 ],
       [3.55827379, 4.16614949, 4.70249877, ..., 5.72599832, 5.65046283,
        5.55649907],
       [3.74999232, 4.35503048, 4.88472607, ..., 6.1678651 , 6.09359214,
        5.99502607]], shape=(5578, 1549))

In [23]:
# Normalize the spectrum
non_normalized_spectrum = spectrum.copy()
spectrum.normalization()

GC-IMS Spectrum: 151207_144657

In [44]:
import numpy as np
np.percentile(spectrum.values, [99.5])

# Inspect some tree parameters here ...
print(f"SPECTRUM INFOS")
print(f"Type: {type(spectrum.values)}")
print(f"Element Type: {type(spectrum.values[0, 0])}")
print("================================================")
print(f"Shape: {spectrum.values.shape}")
num_entries = spectrum.values.shape[0] * spectrum.values.shape[1]
num_non_0_entries = len(np.flatnonzero(spectrum.values))
print(f"Total Entries: {num_entries}")
print(f"Non-0 Enties: {num_non_0_entries}")
print(f"0 Enties: {num_entries - num_non_0_entries}")
print("================================================")
print(f"Max Value: {spectrum.values.max()}")
print(f"Min Value: {spectrum.values.min()}")
print("================================================")
over_50_values = len(np.where(non_normalized_spectrum.values >= 25)[0])
print(f"Total Values: {num_entries}")
print(f"Values > 50: {over_50_values}")
print(f"Values > 50 in % {(over_50_values/num_entries)*100}")
print(f"Percentile Value: {100 - (over_50_values/num_entries)*100}")



SPECTRUM INFOS
Type: <class 'numpy.ndarray'>
Element Type: <class 'numpy.float64'>
Shape: (5578, 1549)
Total Entries: 8640322
Non-0 Enties: 8623185
0 Enties: 17137
Max Value: 1.0
Min Value: 0.0
Total Values: 8640322
Values > 50: 69956
Values > 50 in % 0.8096457516282379
Percentile Value: 99.19035424837176


In [161]:
# Create the tree - Takes roughly 1 min to execute.
from skimage.morphology import max_tree_local_maxima
peak_tree = max_tree_local_maxima(spectrum.values, connectivity=2)


In [18]:
# Detect Peaks with the gc-ims module function
# The fast way .. it works!
spectrum.detect_peaks(threshold_rel=0.05, peak_size=5)


Warning! Make sure that you are using a region without the RIP, for peak detection to function properly.
Warning! Make sure that you are using a region without the RIP, for peak detection to function properly.
Warning! Make sure that you are using a region without the RIP, for peak detection to function properly.
Warning! Make sure that you are using a region without the RIP, for peak detection to function properly.
Warning! Make sure that you are using a region without the RIP, for peak detection to function properly.
Warning! Make sure that you are using a region without the RIP, for peak detection to function properly.
Warning! Make sure that you are using a region without the RIP, for peak detection to function properly.
Warning! Make sure that you are using a region without the RIP, for peak detection to function properly.
Warning! Make sure that you are using a region without the RIP, for peak detection to function properly.
Warning! Make sure that you are using a region without 

GC-IMS Spectrum: 151207_144657

In [21]:
spectrum.peaklist

,peak_label,intensity,volume,abs_dt,ret_time,x,y,n_pixels
peak number,,,,,,,,
1,1,0.059770,0.243505,1.065627,89.523,48,65,47
2,2,0.117340,2.520976,1.019753,94.668,0,100,98
3,3,0.952712,688.167214,1.049380,104.958,31,170,6140
4,4,0.514574,97.702427,1.176491,104.664,164,168,685
5,5,0.059672,0.209777,1.068494,104.664,51,168,47
6,6,0.058970,0.481795,1.152598,115.395,139,241,112
7,7,0.202047,23.130155,1.113413,116.571,98,249,404
8,8,0.068354,1.995297,1.510992,132.006,514,354,242
9,9,0.269104,108.153159,1.114369,133.917,99,367,1588


In [48]:
peak_coords = np.column_stack((spectrum.peaklist['y'].values, spectrum.peaklist['x'].values))

peak_coords[0:5]

array([[ 65,  48],
       [100,   0],
       [170,  31],
       [168, 164],
       [168,  51]])

In [278]:
# Detect Peaks with the gc-imc module function # Takes roughly 3 mins to compute
# The profound way.
spectrum.find_peaks(limit=0.1, denoise=None)
original_peak_table = spectrum.peak_table.copy()

[02-12-2025 14:45:01] [findpeaks.stats] [INFO] Scaling image between [0-255] and to uint8
[02-12-2025 14:45:02] [findpeaks.stats] [INFO] Conversion to gray image.
[02-12-2025 14:45:02] [findpeaks.stats] [INFO] Image is already grayscale, no conversion needed.
[02-12-2025 14:45:02] [findpeaks.stats] [INFO] Detect peaks using topology method with limit at 0.1.
[02-12-2025 14:45:04] [findpeaks.stats] [INFO] Topology
1212583it [00:13, 88792.86it/s]
[02-12-2025 14:45:18] [findpeaks.stats] [INFO] Detect peaks using topology method with limit at 0.1.
[02-12-2025 14:45:25] [findpeaks.stats] [INFO] Topology
8640321it [01:41, 84826.73it/s]


In [322]:
spectrum.peak_table = original_peak_table
original_peak_table

,compound,intensity,abs_dt,x,ret_time,y,birth_level,death_level,score,peak,valley
peak number,,,,,,,,,,,
1,,0.000000,2.499206,1548,899.787,5577,-255.0,-0.0,255.0,False,True
2,,1.000000,1.149730,136,675.906,4054,255.0,0.0,255.0,True,False
3,,0.855098,1.205162,194,433.209,2403,218.0,1.0,217.0,True,False
4,,0.831763,1.170756,158,616.077,3647,212.0,1.0,211.0,True,False
5,,0.801936,1.038867,20,465.990,2626,204.0,3.0,201.0,True,False
...,...,...,...,...,...,...,...,...,...,...,...
4164,,0.003635,1.099077,83,434.238,2410,-255.0,-254.0,1.0,False,True
4165,,0.003813,1.116280,101,432.621,2399,-255.0,-254.0,1.0,False,True
4166,,0.003855,1.295956,289,401.604,2188,-255.0,-254.0,1.0,False,True


In [336]:
np.percentile(spectrum.values, 98.5)

np.float64(0.019748234152188883)

In [318]:
spectrum.peak_table = original_peak_table

filtered_peak_table = original_peak_table.copy()
filtered_peak_table = filtered_peak_table[(original_peak_table['intensity'] > 0.02)]
filtered_peak_table

spectrum.peak_table = filtered_peak_table
spectrum.peak_table

,compound,intensity,abs_dt,x,ret_time,y,birth_level,death_level,score,peak,valley
peak number,,,,,,,,,,,
2,,1.000000,1.149730,136,675.906,4054,255.0,0.0,255.0,True,False
3,,0.855098,1.205162,194,433.209,2403,218.0,1.0,217.0,True,False
4,,0.831763,1.170756,158,616.077,3647,212.0,1.0,211.0,True,False
5,,0.801936,1.038867,20,465.990,2626,204.0,3.0,201.0,True,False
6,,0.723047,1.276841,269,307.671,1549,184.0,1.0,183.0,True,False
...,...,...,...,...,...,...,...,...,...,...,...
4036,,0.063388,1.204207,193,461.433,2595,16.0,15.0,1.0,True,False
4037,,0.125821,1.050336,32,139.062,402,32.0,31.0,1.0,True,False
4038,,0.220699,1.050336,32,124.509,303,56.0,55.0,1.0,True,False


In [19]:
# Plot the all peaks
from matplotlib import pyplot as plt
plt.figure(figsize=(10,6))
spectrum.plot_thresholding(outline_color='red', linewidth=1, annotate=True)
#spectrum.plot_peaks()
plt.xlabel("Drifttime [ms]")
plt.ylabel("Retentiontime [s]")
cbar = plt.gcf().axes[-1]
cbar.set_ylabel("Intensity", fontsize=10)
# Saving the plot image.
image_dir = Path("./output/")
image_name = image_dir / f"Spectrum_peaks.png"
plt.savefig(image_name, dpi=300, bbox_inches='tight')
plt.close()

<Figure size 1000x600 with 0 Axes>

In [17]:
# Plot the spectrum
from matplotlib import pyplot as plt
plt.figure(figsize=(10,6))
#spectrum.plot_thresholding(outline_color='red', linewidth=1, annotate=True)
non_normalized_spectrum.plot()
plt.xlabel("Drifttime [ms]")
plt.ylabel("Retentiontime [s]")
cbar = plt.gcf().axes[-1]
cbar.set_ylabel("Intensity", fontsize=10)
# Saving the plot image.
image_dir = Path("./output/")
image_name = image_dir / f"Spectrum.png"
plt.savefig(image_name, dpi=300, bbox_inches='tight')
plt.close()

<Figure size 1000x600 with 0 Axes>

In [320]:
# Plot peak percistence
from matplotlib import pyplot as plt
plt.figure(figsize=(10,6))
#spectrum.plot_thresholding(outline_color='red', linewidth=1, annotate=True)
spectrum.plot_persistence()
plt.xlabel("Drifttime [ms]")
plt.ylabel("Retentiontime [s]")
cbar = plt.gcf().axes[-1]
cbar.set_ylabel("Intensity", fontsize=10)
# Saving the plot image.
image_dir = Path("./output/")
image_name = image_dir / f"Spectrum_peak_persistence.png"
plt.savefig(image_name, dpi=300, bbox_inches='tight')
plt.close()

In [162]:
# Inspect some tree parameters here ...
print(f"MAX TREE INFOS")
print(f"Type: {type(peak_tree)}")
print(f"Element Type: {type(peak_tree[0, 0])}")
print("================================================")
print(f"Shape: {peak_tree.shape}")
num_entries = peak_tree.shape[0] * peak_tree.shape[1]
num_non_0_entries = len(np.flatnonzero(peak_tree))
print(f"Total Entries: {num_entries}")
print(f"Non-0 Enties: {num_non_0_entries}")
print(f"0 Enties: {num_entries - num_non_0_entries}")
print("================================================")
print(f"Max Tree hierarchy: {peak_tree.max()}")
print(f"Min Tree hierarchy: {peak_tree.min()}")

MAX TREE INFOS
Type: <class 'numpy.ndarray'>
Element Type: <class 'numpy.uint64'>
Shape: (5578, 1549)
Total Entries: 8640322
Non-0 Enties: 169423
0 Enties: 8470899
Max Tree hierarchy: 169423
Min Tree hierarchy: 0


In [185]:
np.percentile(peak_tree, [98.5])
peak_tree_copy = peak_tree.copy()

peak_tree_copy = peak_tree_copy - np.percentile(peak_tree, [99.9])
peak_tree_copy[peak_tree_copy < 0] = 0.0


In [202]:
# Plot a Tree Heatmap
from matplotlib import pyplot as plt
from pathlib import Path
import seaborn as sns

heatmap_dir = Path("./output/")
heatmap_file = heatmap_dir / f"HM_PEAKTREE_.png"
# Create the heatmap plot
plt.figure(figsize=(10, 10))
sns.heatmap(
    peak_tree,
    annot=False,
    fmt=".1f",
    cmap="RdBu_r",
    cbar=True
)
plt.title(f"Peak Heatmap")
plt.xlabel("Drifttime [relative index]")
plt.ylabel("Retentiontime [relative index]")
plt.gca().invert_yaxis()
plt.axis("off")
# Save the heatmap image.
plt.savefig(heatmap_file, dpi=300, bbox_inches="tight")
plt.close()